# 06. Tool 정의
- Tool은 LLM이 호출할 수 있는 함수
- Agent가 Tool을 사용해 외부 데이터나 기능에 접근할 수 있음

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. @tool 데코레이터로 Tool 정의

함수의 docstring이 Tool 설명으로 사용됩니다. LLM이 이 설명을 보고 언제 사용할지 판단함

In [2]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 숫자를 더하는 도구입니다."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱하는 도구입니다."""
    return a * b

print("이름", add.name)
print("설명", add.description)
print("스키마", add.args)
result = add.invoke({"a": 5, "b": 3})
print("결과", result)




이름 add
설명 두 숫자를 더하는 도구입니다.
스키마 {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
결과 8


- StructuredTool 객체에서 자주 쓰는 속성과 메서드  

| 구분  | 속성/메서드        | 용도          | 예시                             |
| --- | ------------- | ----------- | ------------------------------ |
| 속성  | `name`        | 도구 이름 확인    | `add.name`                     |
| 속성  | `description` | 도구 설명 확인    | `add.description`              |
| 속성  | `args`        | 입력 인자 구조 확인 | `add.args`                     |
| 속성  | `args_schema` | 입력 스키마 확인   | `add.args_schema`              |
| 메서드 | `invoke()`    | 도구 직접 실행    | `add.invoke({"a": 1, "b": 2})` |
| 메서드 | `batch()`     | 여러 입력 일괄 실행 | `add.batch([...])`             |
| 메서드 | `ainvoke()`   | 비동기 실행      | `await add.ainvoke({...})`     |


## 2. 실용적인 Tool 예제

In [3]:
from datetime import datetime

@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다. (실제 API 대신 더미 데이터 사용)"""
    weather_data = {
        "서울": "맑음, 22도",
        "부산": "흐림, 18도",
        "제주": "비, 16도"
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

print(get_current_time.invoke({}))
print(get_weather.invoke({"city": "서울"}))

2026-06-09 11:18:26
맑음, 22도


## 3. LLM에 Tool 바인딩 - bind_tools()

LLM이 어떤 Tool을 사용할 수 있는지 알려줍니다.

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LLM이 사용할 수 있는 도구 목록
tools = [add, multiply, get_current_time, get_weather]

# LLM이 Tool을 사용할 수 있도록 래핑 
# LLM은 각 도구의 이름, 설명, 입력값 구조를 참고하여 사용자의 질문에 적절한 도구를 선택하여 호출할 수 있게 됨
llm_with_tools = llm.bind_tools(tools)

# LLM이 Tool 호출을 결정
response = llm_with_tools.invoke("런던 날씨 알려줘.")
#print(response.tool_calls)  # LLM이 어떤 도구를 호출했는지 확인
print(response.content)  # LLM의 최종 응답 출력

```mermaid
sequenceDiagram
    participant User as 사용자
    participant Program as 프로그램
    participant LLM as LLM
    participant Tool as Tool

    User->>Program: "서울 날씨 알려줘"

    Program->>LLM: llm_with_tools.invoke("서울 날씨 알려줘")
    Note over Program,LLM: Tool 목록은 LLM에 전달됨<br/>get_current_time, get_weather, add, multiply

    LLM->>LLM: 질문 분석
    LLM->>LLM: get_weather(city="서울") 호출 필요 판단

    LLM-->>Program: tool_calls 반환
    Note over LLM,Program: name = get_weather<br/>args = {"city": "서울"}

    Program->>Program: print(response.tool_calls)

    Note over Program,Tool: 현재 코드에서는 여기서 Tool을 실행하지 않음
```

In [ ]:
# 수식 계산

response = llm_with_tools.invoke("5 더하기 3은?")
print(response.tool_calls)  # LLM이 어떤 도구를 호출했는지 확인
print(response.content)  # LLM의 최종 응답 출력



[{'name': 'add', 'args': {'a': 5, 'b': 3}, 'id': 'call_GorPaAwvzxnTkl4mS2CIqJCW', 'type': 'tool_call'}]



## 4. Tool 실행 결과를 LLM에 다시 전달

In [12]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = [
    "get_weather", get_weather,
    "get_current_time", get_current_time,
    "add", add,
    "multiply", multiply
]

user_message = HumanMessage(content="런던 현재 날씨 알려줘")
messages = [user_message]

for tool_call in ai_response.tool_calls:
    tool_fn = tool_map[tool_call["name"]]
    print(f"도구 호출: {tool_call['name']} with args {tool_call['args']}")
    tool_result = tool_fn.invoke(tool_call["args"])
    print(f"도구 결과: {tool_call['id']} ")
    messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"]))


final_response = llm_with_tools.invoke(messages)
print(final_response.content)



NameError: name 'ai_response' is not defined

- 시퀀스 다이어그램

```mermaid
sequenceDiagram
    participant User as 사용자
    participant Program as 프로그램
    participant LLM as LLM
    participant Tool as Tool

    User->>Program: 서울 현재 날씨 알려줘

    Program->>LLM: 질문 전달
    LLM-->>Program: get_weather(city="서울") Tool 호출 요청

    Program->>Tool: get_weather.invoke({"city": "서울"})
    Tool-->>Program: "맑음, 22도"

    Program->>LLM: Tool 실행 결과 전달
    LLM-->>Program: 최종 자연어 응답 생성

    Program->>User: 최종 응답 출력
```